# Ray Serve architecture, fault tolerance, and scaling

© 2026, Anyscale. All Rights Reserved

<div class="alert alert-block alert-info">
<b>Roadmap for this notebook</b>
<ol>
    <li>The architecture of a Serve application: controller, proxies, replicas.</li>
    <li>Fault tolerance: health checks, self-healing, and what a crash still loses.</li>
    <li>Scaling across the stack: replicas, Ray nodes, machines.</li>
    <li>How autoscaling works, and the config that drives it.</li>
    <li>How to tune autoscaling: load test, then read the knee.</li>
    <li>Customizing the autoscaling policy.</li>
    <li>BONUS: dynamic batching.</li>
</ol>
</div>

**Imports**

In [ ]:
import asyncio
import time
import threading
from pathlib import Path
from typing import Any, Dict, Tuple

import numpy as np
import ray
import requests
from ray import serve
from ray.serve.config import AutoscalingConfig, AutoscalingContext
from ray.util.state import list_actors
from starlette.requests import Request

## 1. The architecture of a Serve application

Below is a sample Serve instance that we will use to illustrate the architecture.

<img src="https://anyscale-public-materials.s3.us-west-2.amazonaws.com/ray-serve/sample_sercve_instance.png" width="800">

We can break down the above diagram into the following steps:
1. HTTP or GRPC requests come in 
2. The load balancer routes the request to one of the cluster nodes
3. The request is handled by a proxy
4. The proxy routes the request to the relevant deployment replica
5. The replica processes the request and returns the response
6. The proxy returns the response to the client

### 1.1 The three actor types

Serve runs on Ray and utilizes Ray actors.

There are three kinds of actors that are created to make up a Serve instance:

 * **Controller Actor**
    * Global actor unique to each Serve instance
    * Manages the control plane
    * Handles creating, updating, and destroying other actors
    * Runs the Serve Autoscaler
    * Processes Serve API calls for deployment management

  * **Proxy Actors**
    * Run a Uvicorn HTTP server
    * Accept incoming requests
    * Forward requests to replicas
    * Return responses when completed

  * **Replica Actors**
    * Execute the actual request processing code
    * Can host ML models or other business logic
    * Process individual requests from the proxy
    * Support dynamic request batching via `@serve.batch`

Here is a diagram of the Serve architecture:

<img src="https://docs.ray.io/en/latest/_images/architecture-2.0.svg" width="800">

A deployment to look at, split the way any serving code should be: pure `RankerLogic`, unit-testable with no cluster, wrapped in a `Ranker` deployment.

In [ ]:
class RankerLogic:
    def __init__(self, dim: int = 16) -> None:
        self.w = np.ones(dim, dtype="float32")   # stand-in for learned weights
    def score(self, user: list[float], item: list[float]) -> float:
        return float(self.w @ (np.asarray(user) * np.asarray(item)))

@serve.deployment(num_replicas=2)
class Ranker:
    def __init__(self, dim: int = 16) -> None:
        self.model = RankerLogic(dim)             # weights load once per replica
    async def __call__(self, user: list[float], item: list[float]) -> float:
        return self.model.score(user, item)

Deploy it so the dashboard has a live system to show.

In [ ]:
serve.run(Ranker.bind(), name="ranker")

Open the Ray Dashboard's **Serve** tab to see the three actor types live: one controller, a proxy per node, and two `Ranker` replicas.

<img src="https://anyscale-materials.s3.us-west-2.amazonaws.com/ray-serve-deep-dive/serve-dashboard-system-tab.png" loading="lazy" width="700">

<img src="https://anyscale-materials.s3.us-west-2.amazonaws.com/ray-serve-deep-dive/serve-dashboard-applications-tab.png" loading="lazy" width="700">

Tear it down. Each section below starts its own app.

In [ ]:
serve.shutdown()

## 2. Fault tolerance

Serve keeps an application running without you watching it. The pieces:

- **Replica actors restart on failure.** If a replica's process dies, the controller notices the actor is gone and starts a fresh one.
- **Periodic health checks** catch a replica that is alive but wedged. Serve calls an optional `check_health` method on a fixed period; consecutive failures mark the replica unhealthy and it is replaced.
- **Head/GCS fault tolerance** persists the controller's state, so it survives a head-node restart while replicas keep serving.

### 2.1 Health checks

A `DataFetcher` holds a database connection, so its health depends on something outside the replica. Add a `check_health` hook and set the cadence. `check_health` raises if the replica is unusable; otherwise it returns normally.

In [ ]:
# simulate a database connection
def connect_to_db(con_url: str) -> str:
    time.sleep(1)
    return f"Connected to {con_url}"

# write a status file
path =  Path("/mnt/cluster_storage/db_status.txt")
path.write_text("alive")

# check status of connection from file
def is_alive(db: str) -> bool:
    with open(path, "r") as file:
        status = file.read()
    return status == "alive"

@serve.deployment
class DataFetcher:
    def __init__(self, con_url: str) -> None:
        self.db = connect_to_db(con_url)
    
    async def __call__(self, request: Request) -> dict:
        input = await request.json()
        return {"input": input}
    
    async def check_health(self) -> None: # implement health check
        alive = await asyncio.to_thread(is_alive, self.db)
        if not alive:
            raise Exception("Database connection lost")

We run the deployment and expect the health check to pass.

In [ ]:
app_handle = serve.run(DataFetcher.bind("db_url"), name="data-fetcher", blocking=False)

We can simulate a database connection failure by modifying the status file.

In [ ]:
path.write_text("dead")

After three consecutive failed checks (the default threshold), Serve marks the replica `UNHEALTHY` and replaces it. Poll the deployment status to watch the transition; the Ray Dashboard's **Serve** tab and the controller logs show the same thing live.

In [ ]:
for _ in range(60):
    status = serve.status().applications["data-fetcher"].deployments["DataFetcher"].status
    print(status)                                          # HEALTHY -> UNHEALTHY -> HEALTHY (replaced)
    if status == "UNHEALTHY":
        break
    time.sleep(1)

These are the verified defaults Serve applies, straight from the source:

```python
# ray/serve/_private/constants.py (Ray 2.56.0)
DEFAULT_HEALTH_CHECK_PERIOD_S            = 10    # how often check_health runs
DEFAULT_HEALTH_CHECK_TIMEOUT_S           = 30    # a check that hangs this long counts as failed
REPLICA_HEALTH_CHECK_UNHEALTHY_THRESHOLD = 3     # consecutive failures -> restart the replica
PROXY_HEALTH_CHECK_UNHEALTHY_THRESHOLD   = 3     # same, for proxies
DEFAULT_GRACEFUL_SHUTDOWN_TIMEOUT_S      = 20    # drain in-flight requests before killing a replica
DEFAULT_ROLLING_UPDATE_PERCENTAGE        = 0.2   # fraction of replicas replaced per rollout batch
```

In [ ]:
serve.shutdown()

### 2.2 Application errors do not kill the replica

Self-healing covers a dead replica; an *exception in your code* is different. Serve catches it, returns a 500 with the traceback, and the replica stays up to serve the next request.

In [ ]:
@serve.deployment
class FlakyRanker:
    def __init__(self, dim: int = 16) -> None:
        self.model = RankerLogic(dim)
    async def __call__(self, user: list[float], item: list[float]) -> float:
        if len(user) != len(item):
            raise ValueError("user and item must have equal length")   # bug in user code
        return self.model.score(user, item)

handle = serve.run(FlakyRanker.bind(), name="ranker")

In [ ]:
try:
    await handle.remote([1.0] * 16, [0.5] * 8)          # mismatched lengths -> raises in the replica
except Exception as e:
    print("request failed:", type(e).__name__)          # the error propagates to the caller
print(await handle.remote([1.0] * 16, [0.5] * 16))      # same replica still serves: 8.0

Note: the replica is never marked unhealthy for an application error; only failed health checks or a dead process trigger replacement. Over HTTP the same error surfaces as a **500** with the traceback in the body.

In [ ]:
serve.shutdown()

### 2.3 Transient data loss and client retries

Ray serve's controller replaces a dead replica, but any request that was **in flight** on it is lost: the router and replica hold that state in memory, and it does not survive the crash.

Let's create a deployment that simulates a spot instance interruption.

In [ ]:
@serve.deployment
class SpotInstanceReplica:
    def __init__(self) -> None:
        self.state = "my_model"
    async def __call__(self, request: Request) -> str:
        await asyncio.sleep(60)
        return self.state

We run the application

In [ ]:
app = SpotInstanceReplica.bind()
app_handle = serve.run(app, name="spot-instance", blocking=False)

We define some helper functions and class to:
- make a request from a thread
- simulate an instance preemption by killing the replica actor

In [ ]:
def make_request() -> requests.Response:
    response = requests.get("http://localhost:8000/")
    return response


class RequestThread(threading.Thread):
    def run(self) -> None:
        self.result = make_request()


def simulate_instance_preemption() -> None:
    replicas = list_actors(filters=[
        ("state", "=", "ALIVE"),
        ("class_name", "=", "ServeReplica:spot-instance:SpotInstanceReplica"),
    ])
    actor_handle = ray.get_actor(name=replicas[0]["name"], namespace="serve")
    ray.kill(actor_handle)

We now launch two threads:
- one to make a request
- one to simulate an instance preemption

Given we don't have client-side retries, the request will fail immediately after the replica is killed.

In [ ]:
t1 = RequestThread()
t1.start()
time.sleep(20)
t2 = threading.Thread(target=simulate_instance_preemption)
t2.start()

t1.join()
t2.join()

t1_result = t1.result

Serve will replace the replica but all transient requests will be lost.

In [ ]:
t1_result.status_code, t1_result.text

Tear down the spot-instance app so a later section can reuse the `/` route.

In [ ]:
serve.shutdown()

<div class="alert alert-block alert-info">
<b>Best practice.</b> Make clients retry with backoff so a healed replica is transparent. If retries are not an option (non-idempotent work), put a durable queue (Kafka, SQS) in front so requests survive a replica crash.
</div>

### 2.4 Request timeouts

A slow request holds a replica slot and blocks others behind it. An end-to-end HTTP timeout caps that: the proxy terminates a request that runs longer than `request_timeout_s` and returns a 408. It is cluster-global and set in `http_options`, so it is configured at startup, not per deployment.

```yaml
# serve config (cluster-global; not updatable at runtime)
http_options:
  request_timeout_s: 30
```

<div class="alert alert-block alert-warning">
A timed-out request returns <b>408</b>. Pair it with client retries so a transient slow request is retried, not dropped.
</div>

## 3. Scaling across the stack

Adding a replica is one link in a chain. When the autoscaler asks for more replicas than the current nodes can hold, the request cascades down the stack:

- **Serve autoscaler** decides it needs more replicas and asks the controller to start them.
- **Ray cluster autoscaler** sees the pending replica actors as unmet resource demand and provisions more Ray nodes.
- **Kubernetes** (if you run there) schedules each new Ray node as a pod; the **K8s cluster autoscaler** adds machines to the node group when there is no room.

<img src="https://anyscale-public-materials.s3.us-west-2.amazonaws.com/ray-serve/scaling_across_the_stack.png" loading="lazy" width="800">

<div class="alert alert-block alert-info">
Each layer scales independently. A replica only becomes <code>RUNNING</code> once a node exists to place it on, so end-to-end scale-up time includes node (and possibly machine) provisioning, not just replica startup.
</div>

## 4. How autoscaling works

Ray Serve changes the replica count on its own, from traffic. This section builds that loop in order: what gets measured, the policy that turns it into a number, the config you actually set, and the knobs that shape how sharply it reacts.

### 4.1 Replicas report load to the autoscaler

Replicas push their in-flight request counts to the controller, which is where the decision happens.

<img src="https://anyscale-materials.s3.us-west-2.amazonaws.com/ray-serve-deep-dive/serve-autoscaling-replica-reporting.png" width="800">

Note: 
- Deployment handles also send metrics to the autoscaler but are not shown in the diagram above.
- The autoscaling decisions are made within the controller's control loop, not by a separate actor or process, unless you hand the decision to an external scaler (section 6).

### 4.2 The default policy: `target_ongoing_requests`

The same loop, now with the policy that converts reported load into a desired replica count.

<img src="https://anyscale-materials.s3.us-west-2.amazonaws.com/ray-serve-deep-dive/serve_replica_queue_length_autoscaling_policy.png" width="1000">

The primary config is `target_ongoing_requests`

- **Controller will then**:
  - Compare total ongoing requests to `target_ongoing_requests * num_replicas`.
  - Ratio < 1 → scale **down**
  - Ratio > 1 → scale **up**
- **Example**: Ratio = 2 then doubles the replicas.

### 4.3 Set it in code

Pass an `AutoscalingConfig` to the deployment and the loop above is live.

In [ ]:
cfg = AutoscalingConfig(
    min_replicas=1, max_replicas=8,
    target_ongoing_requests=2,                 # hold ~2 in-flight requests per replica
    look_back_period_s=30, upscale_delay_s=30, downscale_delay_s=600,
)
handle = serve.run(
    Ranker.options(autoscaling_config=cfg, max_ongoing_requests=5).bind(),
    name="ranker",
)
serve.status().applications["ranker"].deployments["Ranker"].replica_states   # starts at min_replicas: {'RUNNING': 1}

Note: the deployment comes up at `min_replicas=1`. Under sustained load, `total_ongoing_requests` climbs, `desired` rises above 1, and the controller adds replicas up to `max_replicas`; when load drops, it scales back down after `downscale_delay_s`.

These are the verified `AutoscalingConfig` defaults for a **manual** config:

```python
# AutoscalingConfig() defaults (Ray 2.56.0, ray/serve/config.py):
min_replicas            = 1
max_replicas            = 1      # GOTCHA: a manual config caps at 1, so it never scales unless you raise this
target_ongoing_requests = 2
initial_replicas        = None   # falls back to min_replicas
upscale_delay_s         = 30
downscale_delay_s       = 600
look_back_period_s      = 30
upscaling_factor        = None   # optional damping; None = undamped
downscaling_factor      = None
# smoothing_factor is DEPRECATED. max_ongoing_requests (default 5) is a DEPLOYMENT-level param, NOT a field here.
# Policy (ray/serve/autoscaling_policy.py): desired = total_ongoing_requests / target_ongoing_requests, clamped [min, max].
```

In [ ]:
serve.shutdown()

### 4.4 Shaping the response: delays and factors

`target_ongoing_requests` sets *where* the loop aims. Two other families of knobs set *when* it acts and *how hard* each step is.

The **delays** gate when the loop acts:

- **`upscale_delay_s`** = how long traffic must stay above target before Serve adds replicas. Lower it for fast reaction to spikes.
- **`downscale_delay_s`** = how long traffic must stay below target before Serve removes them. Keep it high so a brief dip does not give back replicas you will need again.
- **`downscale_to_zero_delay_s`** = a separate delay for the 1→0 step; falls back to `downscale_delay_s` if unset.

The **factors** gate how hard each step is:

- **`upscaling_factor`** / **`downscaling_factor`** = a gain on the step size. Above 1 reacts harder, below 1 damps the change to avoid scale up/down thrash.

They come together when you tune for a traffic shape. A bursty workload reacts fast and aggressively on the way up, then gives replicas back slowly:

```python
# tuned for bursty traffic: react in ~5s, scale up aggressively, scale down slowly
AutoscalingConfig(
    min_replicas=1, max_replicas=10, target_ongoing_requests=2,
    upscale_delay_s=5, look_back_period_s=5,    # short window + delay = fast reaction
    downscale_delay_s=600,                       # but slow to give back replicas
    upscaling_factor=1.5,                        # take bigger steps up
)
```

<div class="alert alert-block alert-info">
<code>metrics_interval_s</code> is deprecated, replaced by the env vars <code>RAY_SERVE_REPLICA_AUTOSCALING_METRIC_PUSH_INTERVAL_S</code> (replica load) and <code>RAY_SERVE_HANDLE_AUTOSCALING_METRIC_PUSH_INTERVAL_S</code> (handle queue depth, both 10s). The handle interval drives scale-from-zero responsiveness: a lower value cold-starts sooner.
</div>

## 5. How to tune autoscaling

The defaults give you a working loop; tuning means setting `target_ongoing_requests` to the right number. The reliable way is not arithmetic: load-test one replica and read the setpoint off a Serve metric at the latency knee.

### 5.1 Run a load test

Pin the deployment to one replica and lift `max_ongoing_requests` so Serve's own backpressure does not cap the test below the knee. Then ramp offered RPS with an open-loop generator: each user is capped at one request per second, so offered load stays independent of how slow the service gets. The generator runs outside the notebook, so it is a script, not a cell.

```python
# locustfile.py  (run: locust -f locustfile.py --host http://localhost:8000 --headless --csv ranker)
import random
from locust import HttpUser, task, constant_throughput, LoadTestShape

ITEM_POOL = [f"item_{i}" for i in range(1_000)]

class RankerUser(HttpUser):
    wait_time = constant_throughput(1)           # each user <= 1 req/s; offered RPS ~= active users
    @task
    def rank(self) -> None:
        self.client.post("/rank", name="/rank",
                         json={"user_id": f"u{random.randrange(10_000)}",
                               "candidate_ids": random.sample(ITEM_POOL, 32)})

class RampShape(LoadTestShape):                   # ramp, sustain, spike, then drain
    stages = [(60, 1), (240, 20), (420, 40), (480, 120), (720, 5)]   # (elapsed_s, offered RPS)
    def tick(self) -> tuple[int, float] | None:
        elapsed = self.get_run_time()
        for until_s, users in self.stages:
            if elapsed < until_s:
                return users, 1 / 30
        return None
```

Note: the shape drains at the end on purpose. Scale-up shows within an `upscale_delay_s`, but scale-down only shows once load falls, so a run that stops at peak never exercises half the loop.

### 5.2 Find the threshold and set the config

During the ramp, watch `ray_serve_replica_processing_queries`: this gauge is the replica's ongoing-request count, the exact field the autoscaler divides by `target_ongoing_requests`. Read its value at the offered load where p95 still meets your SLO; that knee reading is what you tune against, no arithmetic.

```promql
avg(ray_serve_replica_processing_queries{deployment="Ranker"})
```

As offered RPS rises, throughput plateaus and latency rises. The measured ongoing-requests-per-replica at that point of latency SLA degradation is the reading you tune against. This is a real load test: P90 latency crosses 500ms at roughly 6 ongoing requests per replica, which suggests `target_ongoing_requests` ~= 4 and `max_ongoing_requests` ~= 6.

<img src="https://anyscale-materials.s3.us-west-2.amazonaws.com/ray-serve-deep-dive/load_test_ongoing_requests.png" loading="lazy" width="1000">

Two buffer ratios turn that knee reading into a config: set `target_ongoing_requests` to about **67-80%** of the knee value (headroom so the autoscaler adds replicas *before* latency degrades), and `max_ongoing_requests` to about **1.2-1.5x** the target (absorbs brief bursts, triggers backpressure if sustained). Size `max_replicas` to peak load over per-replica capacity, plus headroom.

```python
@serve.deployment(
    max_ongoing_requests=8,                                     # ~1.5x target, a burst buffer
    autoscaling_config={"target_ongoing_requests": 5,           # ~75% of the knee read off the gauge
                        "min_replicas": 1, "max_replicas": 10},  # peak_rps / knee_rps + headroom
)
class Ranker: ...
```

## 6. Customizing the autoscaling policy

Serve allows users to customize the autoscaling policy. Below is the verbatim code for the default autoscaling policy that looks at the total in-flight requests and determines the numbers of desired replicas accordingly.

In [ ]:
def inflight_requests_policy(
    ctx: AutoscalingContext,
) -> Tuple[float, Dict[str, Any]]:
    num_running_replicas = ctx.current_num_replicas
    config = ctx.config
    if num_running_replicas == 0:
        return ctx.target_num_replicas, {}
    target_num_requests = config.get_target_ongoing_requests() * num_running_replicas
    error_ratio = ctx.total_num_requests / target_num_requests
    desired_num_replicas = num_running_replicas * error_ratio
    return desired_num_replicas, {}


cfg = AutoscalingConfig(
    min_replicas=1, max_replicas=8,
    policy={"policy_function": inflight_requests_policy},  
)
handle = serve.run(Ranker.options(autoscaling_config=cfg).bind(), name="ranker")
print(serve.status().applications["ranker"].deployments["Ranker"].status)

Note: you express only the target. The framework still applies the bounds, the up/down delays, and the damping factors around your number.

In [ ]:
serve.shutdown()

The custom-policy opens up scaling on signals the default queue-depth loop cannot see:

- **Custom metrics.** Scrape Prometheus (GPU memory, latency percentiles) or export your own from the deployment, and scale on those instead of ongoing requests.
- **Application-level joint scaling.** A policy that receives context for *every* deployment in an app and returns a count for each, to keep ratios between services (a featurizer that must stay 2x a scorer).

To move the decision out of the cluster entirely, opt the application into an external scaler. Serve's own autoscaling must then be off for every deployment in that app, and your own control loop (a custom operator, a capacity planner, a queue-depth monitor) sets the count over REST:

```python
serve.run(Ranker.bind(), name="ranker", external_scaler_enabled=True)

requests.post(
    "http://localhost:8265/api/v1/applications/ranker/deployments/Ranker/scale",
    json={"target_num_replicas": 3},
)
```

Note: the controller is a pure actuator here. It changes the replica count without restarting replicas, and a redeploy resets the count to the config's `num_replicas`.

## 7. BONUS: dynamic batching

A model often runs far more efficiently on a batch than one item at a time. Staggered requests fill a buffer; it flushes either when full or when a timer fires, runs one model call, and fans the results back.

<img src="https://anyscale-public-materials.s3.us-west-2.amazonaws.com/ray-serve-distributed-inference/diagrams/nb2_batch_formation.png" loading="lazy" width="900">

`@serve.batch` coalesces concurrent single-item calls into one batched call, then splits the results back out by position. You write the batched method and an unbatched entry point that awaits it.

In [ ]:
@serve.deployment(max_ongoing_requests=64)
class BatchRanker:
    def __init__(self, dim: int = 16) -> None:
        self.model = RankerLogic(dim)

    @serve.batch(max_batch_size=8, batch_wait_timeout_s=0.1)
    async def handle_batch(self, users: list[list[float]], items: list[list[float]]) -> list[float]:
        print(f"batch size = {len(users)}")                       # watch batches form
        return [self.model.score(u, i) for u, i in zip(users, items)]

    async def __call__(self, user: list[float], item: list[float]) -> float:
        return await self.handle_batch(user, item)

h = serve.run(BatchRanker.bind(), name="ranker")
results = await asyncio.gather(*[h.remote([1.0] * 16, [0.5] * 16) for _ in range(32)])   # 32 concurrent calls
print(results[:3])

Note: each argument to the batched method arrives as a list, and the method returns a list aligned by position. A batch flushes when it reaches `max_batch_size` **or** when `batch_wait_timeout_s` elapses, whichever comes first. (The library defaults are `max_batch_size=10` and `batch_wait_timeout_s=0.01`; here we use 8 and 0.1s to make the batches visible.)

<div class="alert alert-block alert-warning">
A batch can only fill from requests already in flight on the replica, so <code>max_ongoing_requests</code> has to be at least <code>max_batch_size</code>. Left at its default of 5, batches cap at 5 however large <code>max_batch_size</code> is.
</div>

A bigger `max_batch_size` or a longer timeout raises throughput but also tail latency.

In [ ]:
serve.shutdown()

<div class="alert alert-block alert-info">
<b>Reference implementation:</b> the production Ranker lives in <code>code/classic/ranking/ranker.py</code> (with the same <code>RankerLogic</code> vs <code>Ranker</code> split, plus dynamic batching, autoscaling, and live <code>reconfigure</code>) and its paired <code>service.yaml</code>. The load generator below ships as <code>code/classic/load_testing/locustfile.py</code>, with <code>benchmark.yaml</code> for the pinning.
</div>